# 埋め込み（Embeddings）を使用したセマンティックマッチング

**学習目標**
  1. Google Gen AI SDK で Gemini の埋め込みを使用する方法を学ぶ
  2. さまざまな類似度メトリクスを計算する方法を学ぶ
  3. 説明文から生成された埋め込みを使用して、一致する製品を識別する方法を学ぶ
  4. セマンティックマッチングのパフォーマンスを評価する方法を学ぶ
  
セマンティックマッチング（意味的照合）は、エンティティのペア $(x_1, x_2)$ が適切に一致しているかどうかを分類する問題です。したがって、非常に柔軟な分類設定です。具体的には、一般的な情報検索（最初のエンティティがテキストクエリで、2 番目のエンティティが段落など）、エンティティ解決（名寄せ）、データベースレコードのあいまい照合などが含まれます。このノートブックでは、小売製品のテキスト説明のマッチングに焦点を当てます。具体的には以下のとおりです。
  
  
**ユースケースの説明:** あるオンライン小売企業は、自社の在庫商品の価格と競合他社が提供する価格を比較するために Web を巡回しています。彼らの最優先事項は、2 つの製品 Web ページの情報を比較し、Web ページ上の異なる製品説明が同一の製品に対応しているかどうか（これを「マッチ」と呼びます）を示す分類を出力するモデルを実装することです。ここでは、ライプツィヒ大学によって作成された [エンティティ解決ベンチマーク](https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution) の [Amazon-Google Products データセット](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) を使用します。

**モデルの説明:** このノートブックでは、Agent Platform の Gemini Embeddings API を使用して製品説明をマッチングする方法を説明します。[Amazon-GoogleProducts データセット](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) には、Amazon または Google の Web サイトでスクレイピングされた製品に関する製品情報が含まれています。これには製品のタイトル、説明、価格、メーカーが含まれますが、これら 2 つの Web サイトでは表現方法が異なっています。このノートブックでは、製品のタイトルと説明のみを使用してモデルを構築することに焦点を当てます。これらのテキストフィールドを埋め込みに変換し、類似度メトリクスで評価することで、掲載内容の表現がまったく異なる場合でも、製品ペアを高い信頼性でマッチングする方法を学びます。

## セットアップ

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from google import genai
from google.genai import types

pd.options.display.max_colwidth = 1000

In [ ]:
REGION = "us-central1"
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]

# 変更しないでください
os.environ["PROJECT"] = PROJECT
os.environ["REGION"] = REGION

### Text Embeddings API
Text Embeddings API は、テキストを数値ベクトルに変換してセマンティックな意味とコンテキストをキャプチャするモデルを提供します。
推奨されるモデルは `gemini-embedding-001` であり、最大 3072 の出力次元で英語、多言語、コードのタスク全体にわたって最先端のパフォーマンスを提供します。より特殊なユースケースとして、`text-embedding-005` は英語とコードのタスクに特化しており、`text-multilingual-embedding-002` は多言語アプリケーションに焦点を当てています。3 つのモデルすべてが最大 2048 トークンのシーケンス長をサポートしています。
詳細情報:
https://docs.cloud.google.com/gemini-enterprise-agent-platform/reference/models/text-embeddings-api

In [ ]:
EMBEDDING_MODEL = "gemini-embedding-001" 
#EMBEDDING_MODEL = "text-embedding-005"
#EMBEDDING_MODEL = "text-multilingual-embedding-002"

### テキスト埋め込みのタスクタイプ（Task Types）
次の表は、`task_type` パラメータの値とそのユースケースを示しています。

| `Task type` | 説明 |
|---|---|
| **`RETRIEVAL_QUERY`** | 指定されたテキストが検索または取得の設定におけるクエリであることを指定します。ドキュメント側には `RETRIEVAL_DOCUMENT` を使用します。 |
| **`RETRIEVAL_DOCUMENT`** | 指定されたテキストが検索または取得の設定におけるドキュメントであることを指定します。 |
| **`SEMANTIC_SIMILARITY`** | 指定されたテキストがテキスト間のセマンティック類似度（STS: Semantic Textual Similarity）に使用されることを指定します。 |
| **`CLASSIFICATION`** | 埋め込みが分類に使用されることを指定します。 |
| **`CLUSTERING`** | 埋め込みがクラスタリングに使用されることを指定します。 |
| **`QUESTION_ANSWERING`** | クエリの埋め込みが質問への回答に使用されることを指定します。ドキュメント側には `RETRIEVAL_DOCUMENT` を使用します。 |
| **`FACT_VERIFICATION`** | クエリの埋め込みが事実確認（Fact Verification）に使用されることを指定します。ドキュメント側には `RETRIEVAL_DOCUMENT` を使用します。 |
| **`CODE_RETRIEVAL_QUERY`** | クエリの埋め込みが Java および Python のコード検索に使用されることを指定します。ドキュメント側には `RETRIEVAL_DOCUMENT` を使用します。 |

---

### 🔍 検索タスク（Retrieval Tasks）

* **クエリ（Query）**: 入力テキストが検索クエリであることを示すには、`task_type=RETRIEVAL_QUERY` を使用します。
* **コーパス（Corpus）**: 入力テキストが検索対象のドキュメントコレクションの一部であることを示すには、`task_type=RETRIEVAL_DOCUMENT` を使用します。

### 類似度タスク（Similarity Tasks）

* **セマンティック類似度（Semantic similarity）**: 2 つの入力テキストの全体的な意味の類似度を評価するには、両方に `task_type=SEMANTIC_SIMILARITY` を使用します。

In [ ]:
EMBD_TASK_TYPE="SEMANTIC_SIMILARITY"

### クライアントの初期化:
パラメータ `vertexai=True` を指定して Google GenAI クライアントを使用し、Google Cloud プロジェクト ID とリージョンを提供します。

In [ ]:
client = genai.Client(
    vertexai=True,
    project=PROJECT,
    location="global"
)

## 埋め込みの生成
### テキスト文字列から埋め込みを作成するユーティリティメソッドの定義:

In [ ]:
def embed_text(text_to_embed: str | list[str], output_dimensionality: int = None) -> list[list[float]]:
    """提供されたテキストの埋め込みを生成します。"""
    try:
        response = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=text_to_embed,
            config=types.EmbedContentConfig(
                task_type=EMBD_TASK_TYPE,
                output_dimensionality=output_dimensionality 
            )
        )
        
        return [np.array(embed.values) for embed in response.embeddings]

    except Exception as e:
        print(f"Failed to generate embeddings: {e}")

### `embed_text` メソッドの使用例:

In [ ]:
text_to_embed = "emc retrospect 7.5 disk to diskcromwindows"

test_embedding = embed_text(text_to_embed)
print(f"Generated a vector with {len(test_embedding[0])} dimensions.")
print(f"View the first 5 floats: {test_embedding[0][:5]}") # 最初の 5 つの浮動小数点数を表示
input_embd_vector=test_embedding[0]

### 埋め込みベクトルの正規化（L2 ノルム）

`np.linalg.norm()` を使用して L2 ノルム（ユークリッド長）を計算してみましょう。ご覧のとおり、結果のベクトル長は約 `1.0` であり、埋め込みが単位長に正規化されていることが確認できます。正規化された埋め込みを使用すると、2 つのベクトル間のコサイン類似度を単純な内積（ドット積）で直接計算できるため、後続のタスクがシンプルになります。

In [ ]:
magnitude = np.linalg.norm(input_embd_vector)
print(f"vector magnitude = {magnitude}")

### MRL 埋め込みの L2 正規化
**重要な正規化ルール**: API は、デフォルトサイズの出力ベクトルを自動的に L2 正規化します。ただし、`output_dimensionality` を使用してより小さいサイズ（128 や 768 など）を要求した場合、API は正規化されていない未加工の数値を返します。コサイン類似度の代わりに内積を使用したい場合は、コード内で切り捨てられたベクトルを手動で正規化する必要があります。

#### オプション 1: `output_dimensionality` パラメータを使用する:

In [ ]:
embedding_mrl_128 = embed_text(text_to_embed, output_dimensionality=128)
print(f"Generated a vector with {len(embedding_mrl_128[0])} dimensions.")
magnitude = np.linalg.norm(embedding_mrl_128)
print(f"vector magnitude = {magnitude}")

#### オプション 2: 単純に切り捨てる（スライスする）:

In [ ]:
embedding_trunc_128 = input_embd_vector[:128]
print(f"Generated a vector with {len(embedding_trunc_128)} dimensions.")
magnitude = np.linalg.norm(embedding_trunc_128)
print(f"vector magnitude = {magnitude}")

### 埋め込みの L2 正規化

**ベクトル $L_2$ 正規化** は、非ゼロベクトルの合計ユークリッド長（大きさ）が正確に $1$ になるようにスケーリングします。このプロセスにより、任意のベクトルがまったく同じ方向を向いた **単位ベクトル** に変換されます。 

機械学習や埋め込みのワークフローにおいて、ベクトルを単位長に正規化することは非常によくある前処理ステップです。ベクトルが $L_2$ 正規化されている場合、**コサイン類似度** の計算は、計算効率の高い単純な **内積（ドット積）** に簡略化されます。
$L_2$ 正規化の数式は、ベクトル $v$ をそのユークリッド（$L_2$）ノルムで割ります。

$$\hat{v} = \frac{v}{\|v\|_2}$$

#### 1. $L_2$ ノルム（$\|v\|_2$）の計算
分母はユークリッド長であり、各要素の 2 乗の合計の平方根として計算されます。

$$\|v\|_2 = \sqrt{\sum_{i=1}^{n} v_i^2} = \sqrt{v_1^2 + v_2^2 + \dots + v_n^2}$$

#### 2. 要素ごとの除算
スカラーノルムが求まったら、ベクトル内のすべての要素をその値で割ります。

$$\hat{v} = \left[ \frac{v_1}{\|v\|_2}, \frac{v_2}{\|v\|_2}, \dots, \frac{v_n}{\|v\|_2} \right]$$

#### ⚙️ L2 正規化の実装

NumPy を使用すると、この関数の基本的な実装は非常にシンプルです。

In [ ]:
def l2_normalize(vec):
    """ベクトルを単位長（L2 ノルム = 1）に正規化します。"""
    # L2 ノルムを計算
    norm = np.linalg.norm(vec)
    if norm == 0:
        # ゼロ除算の防止
        return vec
    return vec / norm

#### L2 正規化の実装のテスト

In [ ]:
# テスト 1: 標準的な 3D ベクトル（[3, 4] のノルムは 5 なので、正規化後は [0.6, 0.8]）
res1 = l2_normalize([3.0, 4.0])
assert np.allclose(res1, [0.6, 0.8]), f"Failed Standard: Expected [0.6, 0.8], got {res1}"

# テスト 2: 単位ベクトルのチェック（正常に正規化されたベクトルのノルムは正確に 1 でなければならない）
assert np.isclose(np.linalg.norm(res1), 1.0), "Failed Unit Length check"

# テスト 3: ゼロベクトルの安全性（ZeroDivisionError を発生させずにゼロを返す必要がある）
res2 = l2_normalize([0.0, 0.0])
assert np.allclose(res2, [0.0, 0.0]), f"Failed Zero Safety: Expected [0.0, 0.0], got {res2}"

# テスト 4: 既に正規化されているベクトル（変更されないこと）
res3 = l2_normalize([1.0, 0.0])
assert np.allclose(res3, [1.0, 0.0]), f"Failed Already Normalized: Expected [1.0, 0.0], got {res3}"

print("✅ All quick asserts passed successfully!")

L2 正規化後のベクトル長を確認します。

In [ ]:
embedding_l2_norm_mrl_128 = l2_normalize(embedding_trunc_128)
magnitude = np.linalg.norm(embedding_l2_norm_mrl_128)
print(f"vector magnitude = {magnitude}")

### 埋め込みの可視化:

In [ ]:
import plotly.express as px
import numpy as np

# ベクトルを一連のバーとしてプロット
fig = px.bar(
    x=range(len(input_embd_vector)),
    y=input_embd_vector,
    # 重要な変更: 連続的なスケールにマッピングするために値を 'color' に渡す
    color=input_embd_vector, 
    # 発散カラースケールを使用（正は赤、負は青）
    color_continuous_scale="RdBu_r", 
    labels={"x": f"Vector Dimension (0 - {len(input_embd_vector)})", "y": "Activation Value", "color": "Value"},
    title="Vector Visualization (Color-Coded Activation)",
    template="plotly_white"
)

# バーをシャープで明瞭なスタイルに設定
fig.update_traces(marker_line_width=0)
fig.update_layout(
    bargap=0.1,
    coloraxis_cmid=0 
)

fig.show()

### 製品説明を含む最初のデータセットの読み込み

Google および Amazon の Web サイトからスクレイピングされた [製品情報のデータセット](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) を使用します。このデータセットは、ライプツィヒ大学による [セマンティックマッチングとエンティティ解決のベンチマーク](https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution) の一部です。このリポジトリに含まれる次の 3 つのテーブルで構成されています。

```python
../data/Amazon.csv.gz 
../data/GoogleProducts.csv.gz
../Amzon_GoogleProducts_perfectMapping.csv.gz

```

最初のテーブルには、製品のタイトル、説明、メーカーなど、Amazon に掲載されている製品情報が含まれています。

In [ ]:
amazon = pd.read_csv("../data/Amazon.csv.gz")
amazon.columns = [
    "idAmazon",
    "amazon_title",
    "amazon_description",
    "amazon_manufacturer",
    "amazon_price",
]
amazon = amazon.dropna(subset=['amazon_description'])
amazon.head()

### タイトルと説明をスペースで連結して埋め込みを作成する

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Jupyter でプログレスバーを表示するのに便利
import pandas as pd

# 1. バッチサイズを定義
BATCH_SIZE = 20

# タイトルと説明を間にスペースを入れて連結
combined_text = amazon['amazon_title'].fillna('') + " " + amazon['amazon_description'].fillna('')
descriptions = combined_text.tolist()

# 2. 説明を標準の Python リストとして抽出
descriptions = amazon['amazon_description'].tolist()

all_embeddings = []

# 3. バッチを処理して進捗状況を追跡
print(f"Generating embeddings for {len(descriptions)} items in batches of {BATCH_SIZE}...")

for i in tqdm(range(0, len(descriptions), BATCH_SIZE), desc="Embedding Batches"):
    # DataFrame をスライス
    batch_text = descriptions[i : i + BATCH_SIZE]
    
    # バッチの埋め込みを生成
    batch_embeddings = embed_text(batch_text)
    
    # 安全対策: API が空または None を返した場合は空のリストでフォールバック
    if batch_embeddings is None:
        batch_embeddings = []
        
    all_embeddings.extend(batch_embeddings)
    
    # Requests-Per-Minute (RPM) の制限に達するのを防ぐために一時停止
    # 特定の API クォータに基づいてこの値を調整してください
    time.sleep(1.0) 

# 4. 結果の埋め込みを DataFrame に戻して割り当て
if len(all_embeddings) == len(amazon):
    amazon['amazon_embedding'] = all_embeddings
    print("Successfully added embeddings to the DataFrame!")
else:
    print(f"Length mismatch: {len(batch_embeddings)} embeddings for {len(amazon)} rows.")

# 列の幅を一時的に 100 文字に制限
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

## 埋め込み類似度尺度の実装
### ⚙️ 内積（ドット積）の実装
2 つの $n$ 次元ベクトル $A$ と $B$ の **内積（ドット積、スカラー積）** は、各ベクトルの対応する要素同士を掛け合わせ、それらの積を合計することによって計算されます。

$$A \cdot B = \sum_{i=1}^{n} a_i b_i = a_1 b_1 + a_2 b_2 + \dots + a_n b_n$$

ここで:
* $a_i$ と $b_i$ は、インデックス $i$ におけるベクトル $A$ と $B$ の個々の成分（要素）を表します。
* $n$ は次元の総数（埋め込みの長さ）です。

In [ ]:
def calculate_dot_product(vec1, vec2):
    """2 つの埋め込みベクトル間の内積（ドット積）を計算します。"""
    # 対応する要素を zip し、掛け合わせて結果を合計
    #return sum(a * b for a, b in zip(vec1, vec2))
    return np.dot(vec1, vec2)

### 内積の実装のテスト

In [ ]:
# テスト 1: 直交
assert calculate_dot_product([1, 0], [0, 1]) == 0, "Failed Orthogonal Test"
# テスト 2: 標準
assert calculate_dot_product([1, 2, 3], [4, 5, 6]) == 32, "Failed Standard Test"
# テスト 3: 負の値
assert calculate_dot_product([2, 4, -1], [3, -2, 5]) == -7, "Failed Negative Test"
print("✅ All quick tests passed successfully!")

### ⚙️ コサイン類似度の実装

**コサイン類似度** は、ベクトルの大きさに関係なく、2 つのベクトルがどれだけ類似しているかを測定するために使用されるメトリクスです。多次元空間に投影された 2 つのベクトル間の角度のコサインを測定します。 

* 📌 **主な用途:** ドキュメントやテキストの埋め込みを比較するために、自然言語処理（NLP）やレコメンデーションシステムで広く使用されています。
* ⚠️ **値の範囲:** **-1 から 1** の間の値を出力します。**1** はベクトルの方向が同一であることを意味し、**0** は直交（垂直）していることを意味し、**-1** は正反対であることを意味します。

---

#### 数式

2 つのベクトル $\mathbf{A}$ と $\mathbf{B}$ の間のコサイン類似度は次のように計算されます。

$$\text{Cosine Similarity} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$

ここで:
* $\mathbf{A} \cdot \mathbf{B}$ はベクトルの **内積（ドット積）** です。
* $\|\mathbf{A}\|$ と $\|\mathbf{B}\|$ はベクトルの **ユークリッドノルム（大きさ）** です。

---

#### コードの解説

以下は、`calculate_cosine_similarity` 関数の各コンポーネントが数式にどのようにマッピングされているかの概要です。

| ステップ / 行 | コードセグメント | 数学的な対応 | 目的 / 説明 |
| :--- | :--- | :--- | :--- |
| **1. 内積** | `np.dot(embedding1, embedding2)` | $\mathbf{A} \cdot \mathbf{B}$ | 2 つの数値シーケンスの対応する要素の積の合計を計算します。 |
| **2. 大きさ** | `norm(embedding1) * norm(embedding2)` | $\|\mathbf{A}\| \times \|\mathbf{B}\|$ | 両方のベクトルの L2 ノルム（ユークリッド長）の積を計算します。 |
| **3. 安全性チェック** | `if magnitude == 0: return 0.0` | 該当なし | 入力埋め込みのいずれかがすべてゼロのベクトルである場合の **ゼロ除算エラーを回避** します。 |
| **4. 最終的な除算** | `return dot_product / magnitude` | $\frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$ | 内積を大きさの積で割り、最終的なコサイン類似度スコアを取得します。 |

---

#### コードの実装

In [ ]:
from numpy.linalg import norm
def calculate_cosine_similarity(embedding1, embedding2):
    # 内積と大きさを計算
    dot_product = np.dot(embedding1, embedding2)
    magnitude = norm(embedding1) * norm(embedding2)
    
    # 空のベクトルによるゼロ除算を防止
    if magnitude == 0:
        return 0.0

    # 最終的な除算
    return dot_product / magnitude

### コサイン類似度の実装のテスト

In [ ]:
# 1. 同一
assert abs(calculate_cosine_similarity([3, 4], [3, 4]) - 1.0) < 1e-9

# 2. 正反対
assert abs(calculate_cosine_similarity([1, -1], [-1, 1]) - (-1.0)) < 1e-9

# 3. 直交
assert abs(calculate_cosine_similarity([1, 0], [0, 1]) - 0.0) < 1e-9

# 4. ゼロの安全性
assert calculate_cosine_similarity([0, 0], [1, 2]) == 0.0

print("✅ All quick checks passed!")

### ⚙️ ユークリッド距離の実装
**ユークリッド距離** は、ユークリッド空間内の 2 点間の直線距離を計算するために最も一般的に使用されるメトリクスです。 

機械学習や自然言語処理では、2 つのベクトル埋め込み間の非類似度（乖離度）を測定するために使用されます。距離が `0` の場合は埋め込みが同一であることを意味し、値が大きいほど非類似度が大きいことを示します。

---

#### 数式

2 つの $n$ 次元ベクトル $A$ と $B$ について、ユークリッド距離は数学的に次のように定義されます。

$$d(A, B) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

#### ステップごとの解説:
1. **要素ごとの減算 ($a_i - b_i$):** 2 つのベクトルの対応する次元間の差を求めます。
2. **差の 2 乗 ($(a_i - b_i)^2$):** 結果として生じる各差を 2 乗します。これにより負の符号が排除され、より大きな差に対してより重いペナルティが課されます。
3. **総和 ($\sum$):** すべての 2 乗された差を足し合わせます。
4. **平方根 ($\sqrt{x}$):** 距離のスケールを元の単位次元に戻すために、合計の平方根を取ります。

---

#### ユークリッド距離の実装:

In [ ]:
def calculate_euclidean_distance(embedding1, embedding2):
    """2 つの埋め込みベクトル間のユークリッド距離を計算します。
    
    数式: sqrt(sum((A_i - B_i)^2))
    """
    distance = np.sqrt(np.sum((embedding1 - embedding2) ** 2))
    return distance

### ユークリッド距離の実装のテスト

In [ ]:
# 1. 同一性
assert calculate_euclidean_distance(np.asarray([1, 2, 3]), np.asarray([1, 2, 3])) == 0.0

# 2. 既知の 3D 距離 (sqrt(1^2 + 1^2 + 1^2) = sqrt(3) ≈ 1.73205)
expected = 1.7320508075688772
assert abs(calculate_euclidean_distance(np.asarray([1, 1, 1]), np.asarray([2, 2, 2])) - expected) < 1e-9

# 3. 対称性
assert calculate_euclidean_distance(np.asarray([1, 5]), np.asarray([2, 10])) == calculate_euclidean_distance(np.asarray([2, 10]), np.asarray([1, 5]))

print("✅ All quick checks passed successfully!")

### 使用例
1. 検索クエリの埋め込みを生成する（最初の関数を使用）
2. 列全体に類似度関数を適用する
3. 最も関連性の高い結果を先頭に表示するために DataFrame をソートする

In [ ]:
amazon['cosine_similarity'] = amazon['amazon_embedding'].apply(
    lambda row_embedding: calculate_cosine_similarity(input_embd_vector, row_embedding)
)

amazon['dot_product'] = amazon['amazon_embedding'].apply(
    lambda row_embedding: calculate_dot_product(input_embd_vector, row_embedding)
)

amazon['euclidean_distance'] = amazon['amazon_embedding'].apply(
    lambda row_embedding: calculate_euclidean_distance(input_embd_vector, row_embedding)
)

amazon = amazon.sort_values(by='cosine_similarity', ascending=False)
# 列の幅を一時的に 100 文字に制限
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

### 類似度分布の可視化
類似度分布のヒストグラムをプロットします

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# クリーンな視覚スタイルを設定
sns.set_theme(style="whitegrid")

# 図を初期化
plt.figure(figsize=(10, 6))

sns.histplot(
    amazon['cosine_similarity'],
    bins=100,
    kde=False,
    color='royalblue'
)

# 読みやすさのためにタイトルとラベルを追加
plt.title('Distribution of Cosine Similarity', fontsize=14, pad=15)
plt.xlabel('Cosine Similarity', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

# レイアウトを調整して表示
plt.tight_layout()
plt.show()

In [ ]:
# クリーンな視覚スタイルを設定

# 図を初期化
plt.figure(figsize=(10, 6))

sns.histplot(
    amazon['dot_product'],
    bins=100,
    kde=False,
    color='royalblue'
)

# 読みやすさのためにタイトルとラベルを追加
plt.title('Distribution of Dot Product', fontsize=14, pad=15)
plt.xlabel('Dot Product', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

# レイアウトを調整して表示
plt.tight_layout()
plt.show()

In [ ]:
# クリーンな視覚スタイルを設定

# 図を初期化
plt.figure(figsize=(10, 6))

sns.histplot(
    amazon['euclidean_distance'],
    bins=100,
    kde=False,
    color='royalblue'
)

# 読みやすさのためにタイトルとラベルを追加
plt.title('Distribution of Euclidean Distance', fontsize=14, pad=15)
plt.xlabel('Euclidean Distance', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

# レイアウトを調整して表示
plt.tight_layout()
plt.show()

### cosine_similarity 列の削除

In [ ]:
amazon = amazon.drop(columns=["cosine_similarity"])

### 切り捨てられたベクトル（MRL）の追加

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Jupyter でプログレスバーを表示するのに便利
import pandas as pd

# 埋め込みを 128 次元に切り捨て、結果を "amazon_embedding_128" 列として保存
amazon['amazon_embedding_128'] = [
    embed[:128] for embed in amazon['amazon_embedding']
]

# 切り捨てられたベクトルに L2 正規化を適用し、結果を "amazon_embedding_l2norm_128" として保存
amazon['amazon_embedding_l2norm_128'] = [
    l2_normalize(embed) for embed in amazon['amazon_embedding_128']
]

# 列の幅を一時的に 100 文字に制限
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

## 製品ペアを見つけるためのセマンティックマッチングの実装
### 2 つ目のデータセットの読み込み
2 番目のテーブルには、Google の Web サイトからスクレイピングされた製品に関する同じ情報が含まれています。

In [ ]:
google = pd.read_csv("../data/GoogleProducts.csv.gz")
google.columns = [
    "idGoogleBase",
    "google_title",
    "google_description",
    "google_manufacturer",
    "google_price",
]
google = google.dropna(subset=['google_description'])
google.head()

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Jupyter でプログレスバーを表示するのに便利

# 1. バッチサイズを定義
BATCH_SIZE = 20

# 2. 説明を標準の Python リストとして抽出
#descriptions = google['google_description'].tolist()
combined_text = google['google_title'].fillna('') + " " + google['google_description'].fillna('')
descriptions = combined_text.tolist()

all_embeddings = []

# 3. バッチを処理して進捗状況を追跡
print(f"Generating embeddings for {len(descriptions)} items in batches of {BATCH_SIZE}...")

for i in tqdm(range(0, len(descriptions), BATCH_SIZE), desc="Embedding Batches"):
    # DataFrame をスライス
    batch_text = descriptions[i : i + BATCH_SIZE]
    
    # バッチの埋め込みを生成
    batch_embeddings = embed_text(batch_text)
    
    # 安全対策: API が空または None を返した場合は空のリストでフォールバック
    if batch_embeddings is None:
        batch_embeddings = []
        
    all_embeddings.extend(batch_embeddings)
    
    # Requests-Per-Minute (RPM) の制限に達するのを防ぐために一時停止
    # 特定の API クォータに基づいてこの値を調整してください
    time.sleep(1.0) 

# 4. 結果の埋め込みを DataFrame に戻して割り当て
if len(all_embeddings) == len(google):
    google['google_embedding'] = all_embeddings
    print("Successfully added embeddings to the DataFrame!")
else:
    print(f"Length mismatch: {len(batch_embeddings)} embeddings for {len(google)} rows.")

# 列の幅を一時的に 100 文字に制限
with pd.option_context('display.max_colwidth', 100):
    display(google.head())

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Jupyter でプログレスバーを表示するのに便利
import pandas as pd

# 埋め込みを 128 次元に切り捨て、結果を "google_embedding_128" 列として保存
google['google_embedding_128'] = [
    embed[:128] for embed in google['google_embedding']
]

# 切り捨てられたベクトルに L2 正規化を適用し、結果を "google_embedding_l2norm_128" として保存
google['google_embedding_l2norm_128'] = [
    l2_normalize(embed) for embed in google['google_embedding_128']
]

# 列の幅を一時的に 100 文字に制限
with pd.option_context('display.max_colwidth', 100):
    display(google.head())

### グラウンドトゥルース（正解）データセットの読み込み

最後のテーブルには、両方の Web サイトで同じ製品に対応する製品情報（ただし、2 つの Web サイトで説明が異なる場合があります）のマッチングが含まれています。

このデータセットを使用して、セマンティックマッチングのアプローチを評価します。
`matching` DataFrame 内の一致する製品 ID のサンプル（サイズは `SAMPLE_SIZE` で制御）を取得し、`google` および `amazon` DataFrame に含まれる Google と Amazon の製品情報を結合します。

In [ ]:
matching = pd.read_csv("../data/Amzon_GoogleProducts_perfectMapping.csv.gz")
SAMPLE_SIZE=1000 # 最初の 1000 件のマッチした製品ペアのみを使用
matching = matching.merge(right=google, how="left", on="idGoogleBase")
matched_products = matching.merge(right=amazon, how="left", on="idAmazon")
matched_products = matched_products.dropna(subset=['amazon_embedding', 'google_embedding']).head(SAMPLE_SIZE)
matched_products.head()

## 📊 マッチング結果の評価とランキング

**`evaluate_matching_ranked`** は、反復的なペアワイズ比較アプローチを使用してエンティティマッチングのパフォーマンス（Amazon の製品リストと Google の製品リストのペアリングなど）を評価し、**Top-K 精度** メトリクスを計算します。

---

### 主要なアルゴリズムの概念

#### 1. ペアワイズ比較行列（計算量 $O(N \times M)$）
$N$ 個の Amazon アイテムと $M$ 個の Google アイテムを含むデータセットについて、この関数はサイズ $N \times M$ の 2D グリッド（行列）を構築します。Amazon データセット内のすべてのアイテムは、ユーザー定義の `similarity_func` を使用して Google データセット内のすべてのアイテムと体系的に比較されます。

$$\mathbf{S}_{i, j} = \text{similarity\_func}(\text{Amazon}_i, \text{Google}_j)$$

#### 2. ソート戦略
ソートの動作は、メトリクスが **距離**（値が小さいほど良い）か **類似度**（値が大きいほど良い）かに基づいて動的に適応します。

* **距離ベース（$is\_distance=True$）:** インデックスを **昇順** でランク付けします。
* **類似度ベース（$is\_distance=False$）:** インデックスを **降順** でランク付けします（負の行列反転 `-similarity_matrix` を使用して実現）。

---

### パラメータ設定

| パラメータ | 型 | デフォルト | 説明 |
| :--- | :--- | :--- | :--- |
| **`df`** | *pandas.DataFrame* | *必須* | 並列データストリーム（正解ペアや識別子など）を含む入力 DataFrame。 |
| **`col_amazon`** | *str* | *必須* | Amazon のテキスト / 埋め込みを含む DataFrame の列名。 |
| **`col_google`** | *str* | *必須* | Google のテキスト / 埋め込みを含む DataFrame の列名。 |
| **`similarity_func`** | *function* | *必須* | アイテムの比較に使用されるスコアリング関数（コサイン類似度、レーベンシュタイン距離など）。 |
| **`is_distance`** | *bool* | `False` | ソートの方向を決定します。距離メトリクス（ユークリッド距離など）の場合は `True`、類似度メトリクス（内積など）の場合は `False` に設定します。 |

---

### コード実行の内訳

アルゴリズムは、5 つの明確な操作ステップでマッチング問題を処理します。

| ステップ | 操作 | 関連する主要変数 | 技術的な目的 |
| :--- | :--- | :--- | :--- |
| **1** | **初期化とセットアップ** | `amazon_data`, `google_data`, `google_ids` | 入力 DataFrame をコピーし、イテレーション中のメモリメモリアクセスを高速化するためにターゲットを軽量な Python リストに抽出します。 |
| **2** | **完全反復行列** | `similarity_matrix` | ネストされたループを実行して、`similarity_func` を正確に $N \times M$ 回実行します。`time.perf_counter()` を使用してタイムスタンプを記録します。 |
| **3** | **インデックスのランキング** | `ranked_indices` | `axis=1` に沿って `np.argsort` を使用してマッチをランク付けします。距離の場合は値が小さいものが上位になり、類似度の場合は値が大きいものが上位になります。 |
| **4** | **行ごとの評価** | `all_ranked_ids`, `correct_match_ranks` | 各 Amazon の行を反復処理して、真のターゲット Google ID がどこにランク付けされているかを見つけ、未加工の配列インデックスを **1 から始まるランク（1-based Rank）** に変換します。 |
| **5** | **メトリクスと出力の生成** | `top_1_acc` 〜 `top_20_acc` | Top-K 精度（真の一致が上位 $K$ 個の選択肢に含まれていた割合）を計算し、コンソールに統計を出力します。 |

---

### 出力メトリクスの解釈

* **最良マッチ精度（Top-1）:** モデルの絶対的な第 1 位の推奨が真のターゲットの一致であったインスタンスの割合。
* **Top-K 精度（$K \in \{3, 5, 10, 20\}$）:** 正しい一致が上位 $K$ 個のランク予測に含まれていたインスタンスの割合。
  
$$\text{Top-}K\text{ 精度} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{I}(\text{Rank}_i \le K) \times 100$$

*(ここで $\mathbb{I}$ は条件が真の場合に 1、それ以外の場合は 0 を返す指示関数です)*

In [ ]:
import time
import numpy as np
import pandas as pd

def evaluate_matching_ranked(df, col_amazon, col_google, similarity_func, is_distance=False):
    """Amazon と Google の列間のマッチングを評価し、すべての組み合わせをランク付けします。
    
    戻り値:
    - result_df: ランクリスト（1 〜 N）と Top-K メトリクスが付加された DataFrame。
    """
    result_df = df.copy()
    
    amazon_data = result_df[col_amazon].tolist()
    google_data = result_df[col_google].tolist()
    
    # 予測にマッピングするために正解 ID を抽出
    google_ids = result_df['idGoogleBase'].values 
    
    N = len(amazon_data)
    M = len(google_data)
    
    metric_type = "distance" if is_distance else "similarity"
    print(f"Calculating {metric_type} iteratively for {N} x {M} ({N * M:,}) combinations...")
    
    similarity_matrix = np.zeros((N, M))
    start_time = time.perf_counter()

    # --- 完全反復ロジック ---
    # すべての Amazon アイテムとすべての Google アイテムを比較
    for i in range(N):
        for j in range(M): 
            similarity_matrix[i, j] = similarity_func(amazon_data[i], google_data[j])
            
    end_time = time.perf_counter()

    # --- パフォーマンスメトリクスの計算 ---
    execution_time = end_time - start_time
    
    # --- ランキングロジック ---
    # np.argsort は配列をソートするインデックスを返す
    if is_distance:
        # 距離の場合、小さいほど良い（昇順ソート）
        ranked_indices = np.argsort(similarity_matrix, axis=1)
    else:
        # 類似度の場合、大きいほど良い（降順ソート）
        # 高い順にソートするために負の similarity_matrix を使用
        ranked_indices = np.argsort(-similarity_matrix, axis=1)

    # 新しい DataFrame 列用のリストを事前割り当て
    all_ranked_ids = []
    all_ranked_scores = []
    correct_match_ranks = []

    for i in range(N):
        # この特定の Amazon 行のソート済みインデックスを抽出
        row_sorted_indices = ranked_indices[i]
        
        # ソートされたインデックスに基づいて ID とスコアを並べ替え
        sorted_scores = similarity_matrix[i, row_sorted_indices]
        sorted_ids = google_ids[row_sorted_indices]
        
        all_ranked_ids.append(sorted_ids.tolist())
        all_ranked_scores.append(sorted_scores.tolist())
        
        # 実際の正解 ID が何位にランクされたかを計算（1 ベースのインデックス）
        true_id = result_df['idGoogleBase'].iloc[i]
        rank_pos = np.where(sorted_ids == true_id)[0]
        
        if len(rank_pos) > 0:
            correct_match_ranks.append(rank_pos[0] + 1) # ランクが 0 ではなく 1 から始まるように +1
        else:
            correct_match_ranks.append(None)
            
    # --- DataFrame への割り当て ---
    result_df['ranked_predicted_idGoogleBase'] = all_ranked_ids
    result_df['ranked_scores'] = all_ranked_scores
    result_df['correct_match_rank'] = correct_match_ranks
    
    # 元のロジックを保持: 1 位のベストマッチを分離
    result_df['predicted_idGoogleBase'] = [ids[0] for ids in all_ranked_ids]
    result_df['best_match_score'] = [scores[0] for scores in all_ranked_scores]
    result_df['is_correct_match'] = result_df['idGoogleBase'] == result_df['predicted_idGoogleBase']
    
    # 行列が完全に正方である場合にのみ対角ペアスコアを計算
    if N == M:
        result_df['pair_score'] = similarity_matrix.diagonal()
    else:
        result_df['pair_score'] = None
    
    total_comparisons = N * M
    
    # Top-K 精度を計算
    top_1_acc = (result_df['correct_match_rank'] == 1).mean() * 100
    top_3_acc = (result_df['correct_match_rank'] <= 3).mean() * 100
    top_5_acc = (result_df['correct_match_rank'] <= 5).mean() * 100
    top_10_acc = (result_df['correct_match_rank'] <= 10).mean() * 100
    top_20_acc = (result_df['correct_match_rank'] <= 20).mean() * 100
    
    print("-" * 40)
    print(f"Execution time: {execution_time:.4f} seconds")
    print(f"Total pairwise comparisons calculated: {total_comparisons:,}")
    print(f"Average time per comparison: {(execution_time / total_comparisons) * 1e6:.2f} microseconds")
    print("-" * 40)
    print("Model Accuracy (Where did the correct match rank?):")
    print(f"Best Match Accuracy: {top_1_acc:.2f}%")
    print(f"Top-3 Accuracy: {top_3_acc:.2f}%")
    print(f"Top-5 Accuracy: {top_5_acc:.2f}%")
    print(f"Top-10 Accuracy: {top_10_acc:.2f}%")
    print(f"Top-20 Accuracy: {top_20_acc:.2f}%")
    print("-" * 40)
    
    return result_df

### 実験: セマンティックマッチングの構成

コードは、ペアごとに評価パイプライン `evaluate_matching_ranked` を呼び出すことにより、**埋め込み** と **距離 / 類似度メトリクス** のすべての組み合わせを体系的に評価します。

In [ ]:
import time
import pandas as pd
import numpy as np

# 構成設定を定義
embd_configs = ["embedding", "embedding_128", "embedding_l2norm_128"]

# 関数オブジェクトとわかりやすい名前および距離プロパティをペアリング
similarity_configs = [
    {"func": calculate_euclidean_distance, "name": "Euclidean Distance", "is_distance": True},
    {"func": calculate_cosine_similarity, "name": "Cosine Similarity", "is_distance": False},
    {"func": calculate_dot_product, "name": "Dot Product", "is_distance": False}
]

# 各実行の統計を収集するリスト
benchmark_results = []

for embd_config in embd_configs:
    for sim_cfg in similarity_configs:
        sim_func = sim_cfg["func"]
        sim_name = sim_cfg["name"]
        is_distance = sim_cfg["is_distance"]
        
        config_label = f"{embd_config} + {sim_name}"
        print(f"\n[RUNNING] Configuration: {config_label}")
        
        # 呼び出しの全体的な実行時間を追跡
        start_time = time.perf_counter()
        
        # 評価メソッドを実行
        evaluated_df = evaluate_matching_ranked(
            df=matched_products, 
            col_amazon=f'amazon_{embd_config}', 
            col_google=f'google_{embd_config}', 
            similarity_func=sim_func,
            is_distance=is_distance
        )
        
        total_time = time.perf_counter() - start_time
        
        # 返された DataFrame から直接パフォーマンスメトリクスを取得
        top_1_acc = (evaluated_df['correct_match_rank'] == 1).mean() * 100
        top_3_acc = (evaluated_df['correct_match_rank'] <= 3).mean() * 100
        top_5_acc = (evaluated_df['correct_match_rank'] <= 5).mean() * 100
        top_10_acc = (evaluated_df['correct_match_rank'] <= 10).mean() * 100
        top_20_acc = (evaluated_df['correct_match_rank'] <= 20).mean() * 100
        
        # すべてのメタデータとメトリクスを追加
        benchmark_results.append({
            "Embedding Config": embd_config,
            "Similarity Metric": sim_name,
            "Configuration": config_label,
            "Top-1 Accuracy (%)": top_1_acc,
            "Top-3 Accuracy (%)": top_3_acc,
            "Top-5 Accuracy (%)": top_5_acc,
            "Top-10 Accuracy (%)": top_10_acc,
            "Top-20 Accuracy (%)": top_20_acc,
            "Execution Time (s)": total_time
        })

# 最終的な比較用 DataFrame を作成
comparison_df = pd.DataFrame(benchmark_results)
print("\n[SUCCESS] All configurations evaluated successfully!")

### 単一の埋め込み / 類似度構成ごとに Top-1 から Top-20 への精度の推移を可視化します。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_all_configs_grid(df):
    """個別のグラフを 3x3 グリッドでプロットします。各グラフは、
    単一の埋め込み / 類似度構成に対する Top-1 から Top-20 の精度の推移を示します。
    """
    sns.set_theme(style="whitegrid")
    
    # 一意の構成を特定（3 つの埋め込み構成 * 3 つの類似度構成 = 合計 9 つ）
    configs = df["Configuration"].unique()
    num_configs = len(configs)
    
    # グリッドの次元を動的に計算
    cols = 3
    rows = (num_configs + cols - 1) // cols
    
    # クリーンで広々とした図レイアウトを初期化
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows), sharey=True)
    axes = axes.flatten()  # 反復しやすいようにグリッドを 1D 配列にフラット化
    
    # メトリクスと DataFrame 内の対応する列を定義
    metrics = ["Top-1", "Top-3", "Top-5", "Top-10", "Top-20"]
    metric_cols = [f"{m} Accuracy (%)" for m in metrics]
    
    for i, config in enumerate(configs):
        ax = axes[i]
        
        # この特定の構成に属する行を分離
        config_row = df[df["Configuration"] == config]
        accuracy_values = config_row[metric_cols].values[0]
        
        # 連続バーチャートをプロット
        sns.barplot(x=metrics, y=accuracy_values, ax=ax, hue=metrics, palette="crest", legend=False)
        
        # サブプロットのカスタマイズ
        ax.set_title(config, fontsize=12, fontweight='bold', pad=10)
        ax.set_ylim(0, 115)
        ax.set_ylabel("Accuracy (%)" if i % cols == 0 else "")  # Y 軸ラベルは左側の列のみ
        
        ax.set_xlabel("Accuracy Threshold", fontsize=9, fontweight='bold')
        
        # 各バーの真上に値ラベルを描画
        for bar in ax.patches:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.0, 
                height + 2, 
                f'{height:.1f}%', 
                ha='center', 
                va='bottom', 
                fontsize=9, 
                fontweight='bold'
            )
            
    # グリッドサイズが不均一な場合の余分な空のサブプロットを削除
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.suptitle("Top-K Accuracy Progression per Configuration", fontsize=20, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.show()

# グリッドビジュアライザーを実行
plot_all_configs_grid(comparison_df)

### 構成の組み合わせ全体で精度と実行時間を比較する

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_configuration_dashboard_vertical(df):
    """垂直に積み重ねられたサブプロットを持つデュアルプロットダッシュボードを生成し、
    構成の組み合わせ全体で精度と実行時間を比較します。
    """
    # クリーンでモダンな美観を適用
    sns.set_theme(style="whitegrid")
    
    # 2 行 1 列に再構成
    fig, axes = plt.subplots(2, 1, figsize=(12, 16))
    
    # --- プロット 1: 複数メトリクス精度の比較（上部プロット） ---
    accuracy_cols = ["Top-1 Accuracy (%)", "Top-5 Accuracy (%)", "Top-20 Accuracy (%)"]
    df_melted = df.melt(
        id_vars=["Configuration"],
        value_vars=accuracy_cols,
        var_name="Accuracy Metric",
        value_name="Accuracy (%)"
    )
    
    sns.barplot(
        ax=axes[0],
        data=df_melted,
        x="Configuration",
        y="Accuracy (%)",
        hue="Accuracy Metric",
        palette="Blues_d"
    )
    
    axes[0].set_title("Accuracy Metrics by Configuration", fontsize=16, fontweight='bold', pad=15)
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Accuracy (%)", fontsize=12, fontweight='bold')
    axes[0].set_ylim(0, 115)
    axes[0].tick_params(axis='x', rotation=30, labelsize=10)
    
    # バーの上に精度パーセンテージを注記
    for container in axes[0].containers:
        axes[0].bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

    # --- プロット 2: 計算速度（下部プロット） ---
    sns.barplot(
        ax=axes[1],
        data=df,
        x="Configuration",
        y="Execution Time (s)",
        hue="Configuration",
        palette="Reds_d",
        legend=False
    )
    
    axes[1].set_title("Execution Time Comparison (Lower is Better)", fontsize=16, fontweight='bold', pad=15)
    axes[1].set_xlabel("Configuration Combo", fontsize=12, fontweight='bold', labelpad=15)
    axes[1].set_ylabel("Execution Time (Seconds)", fontsize=12, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=30, labelsize=10)
    
    # バーの上に秒数を注記
    for container in axes[1].containers:
        axes[1].bar_label(container, fmt='%.3fs', padding=3, fontsize=10)

    # レイアウトを整えて表示
    plt.tight_layout()
    plt.show()

# 垂直ビジュアライゼーションダッシュボードを実行
plot_configuration_dashboard_vertical(comparison_df)

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.